In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import tool
from typing import TypedDict, Union



@tool
def add_tool(a: float, b: float) -> float:
    """Adds two numbers."""
    return a + b

@tool
def multiply_tool(a: float, b: float) -> float:
    """Multiplies two numbers."""
    return a * b


tools = {
    "add": add_tool,
    "multiply": multiply_tool,
}
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")



def tool_node(tool_name):
    def executor(state):
        user_input = state["input"]
        # Extract numbers
        import re
        nums = list(map(float, re.findall(r"[-+]?\d*\.\d+|\d+", user_input)))
        if len(nums) != 2:
            return {"result": "Please provide exactly two numbers."}
        result = tools[tool_name](a=nums[0], b=nums[1])
        return {"result": result}
    return executor

class GraphState(TypedDict):
    input: str
    result: Union[str, float, None]
    next: str  # "add" or "multiply"
def router_node(state: GraphState) -> GraphState:
    user_input = state["input"]
    response = llm.invoke(
        f"Decide whether to use addition or multiplication for this input: '{user_input}'. "
        f"Only respond with 'add' or 'multiply'."
    )
    decision = response.content.strip().lower()
    if decision not in tools:
        decision = "add"  
    return {**state, "next": decision}


graph = StateGraph(GraphState)

graph.add_node("router", RunnableLambda(router_node))
graph.add_node("add", RunnableLambda(tool_node("add")))
graph.add_node("multiply", RunnableLambda(tool_node("multiply")))

graph.set_entry_point("router")

graph.add_conditional_edges("router", lambda state: state["next"], {
    "add": "add",
    "multiply": "multiply",
})


graph.add_edge("add", END)
graph.add_edge("multiply", END)


executable = graph.compile()


result = executable.invoke({"input": "Multiply 3 and 9", "result": None, "next": ""})
print(result)


C:\Users\Parthiban\AppData\Local\Temp\ipykernel_4372\510498566.py:57: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = tools[tool_name](a=nums[0], b=nums[1])


TypeError: BaseTool.__call__() got an unexpected keyword argument 'a'